In [ ]:
import sys, kardemumma
print(sys.executable)         # kernel Python
print(kardemumma.__file__)    # should point to your local repo under src/kardemumma

In [ ]:
# Load packages
from pathlib import Path
import os
import re
import sys
import importlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyteomics import achrom
# Importantly, import kardemumma :)
import kardemumma as kdm

In [ ]:

# Skyline data path
skyline_path = "data/DA4000/DA4k_p1-8.csv"

# Import skyline data (includes column "Isotope Label Type" from Precursor)
skyline_importer = kdm.ImportSkylineFile(skyline_path)
skyline_data = skyline_importer.import_skyline_file()

In [ ]:
# **SDRF**
sdrf_path = 'data/DA4000/20260523_DA4K.sdrf.tsv'

# Valdate sdrf fasta_file
# kdm.validate_sdrf(sdrf_path)
kdm.readout_ms_type(sdrf_path)
# Import SDRF file for this project

sdrf_data = kdm.ImportSDRFFile(sdrf_path)
sdrf_data_file = sdrf_data.import_sdrf_file()



In [ ]:
# # # Valdate sdrf 
# kdm.validate_sdrf(sdrf_path)

In [ ]:
# Print out what Peptide Sequence iRT

kdm.get_irt_peptides(skyline_data)

In [ ]:
# Checking possible QC samples from Skyline results

qc_samples = skyline_importer.suggest_qc_samples(skyline_data)
# skyline_checker = CheckSkylineFile(skyline_path)

# Suggest qc samples

print(qc_samples)


In [ ]:
# # Plot retention time

# skyline_rt = skyline_data.copy()

# # Calculate mean retention time for each peptide
# mean_rt = skyline_data.groupby('Peptide Sequence')['Predicted Retention Time'].mean()

# # Sort peptides by mean retention time
# sorted_peptides = mean_rt.sort_values().index.tolist()

# # Calculate predicted RT using pyteomics.achrom's calculate_RT with Guo coefficients
# skyline_rt['RT'] = skyline_rt['Peptide Sequence'].apply(lambda seq: achrom.calculate_RT(seq, achrom.RCs_guo_ph7_0))

# skyline_rt.head()

In [ ]:
# # Plot matching between predicted RT and experimental Predicted Retention Time

# plt.figure(figsize=(8, 6))
# plt.scatter(
#     skyline_rt['RT'],
#     skyline_rt['Predicted Retention Time'],
#     alpha=0.6,
#     s=20
# )
# plt.xlabel('Predicted RT (achrom Guo, pH 7.0)')
# plt.ylabel('Experimental RT (Skyline (Koina): Predicted Retention Time)')
# plt.title('Predicted vs. Experimental Retention Time for Peptides')
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [ ]:
# Remove QC samples from skyline_data
skyline_data = kdm.remove_qc_samples(skyline_data, qc_samples)


In [ ]:
kdm.plot_library_dot_product_distribution(skyline_data)

In [ ]:
skyline_clean = kdm.filter_library_dot_product(skyline_data, threshold=0.6)


In [ ]:
# Summarise each peptide to count how many heavy or light signals are present in skyline_pivot
peptide_counts = kdm.summarise_peptide_counts(skyline_clean)


In [ ]:
report_summary, peptide_list = kdm.report_peptide_protein_summary(peptide_counts)

In [ ]:
kdm.plot_heavy_light_scatter(peptide_counts)

In [ ]:
filtered_peptide_counts = kdm.filter_peptide_counts(peptide_counts, light_cutoff=600, heavy_cutoff=600)

filtered_peptide_counts.head()


In [ ]:
selected_peptides_report, selected_peptides = kdm.report_peptide_protein_summary(filtered_peptide_counts)

In [ ]:
# from kardemumma.importer import MergeFiles

skyline_merge_obj = kdm.MergeFiles(skyline_data, sdrf_data_file, selected_peptides)
skyline_merge = skyline_merge_obj.merge_files()

In [ ]:
skyline_merge.head(20)

In [ ]:
skyline_pool = skyline_merge_obj.select_pool_data(col_sample='characteristics[Sample]', sample_value='PlasmaPool')
# Need to check printing????

In [ ]:
skyline_pool.head()

In [ ]:
# Plot log_ratio of each sample in boxplot, colored by plate, but x-axis is Replicate, sorted by plate (though x labels are hidden).
kdm.plot_pool_boxplot(skyline_pool)


In [ ]:
# Calculate intra-plate CV

peptide_plate_stats = kdm.calculate_intra_plate_cv(skyline_pool, col_name='characteristics[plate]')
kdm.plot_intra_plate_cv_stats(peptide_plate_stats, col_name='characteristics[plate]')

In [ ]:
interplate_cv = kdm.calculate_inter_plate_cv(peptide_plate_stats)
kdm.plot_inter_plate_cv_kde(interplate_cv)

In [ ]:
interplate_cv.head()

In [ ]:
# Print interplate_cv froom low to high
print('This is the interplate_cv from low to high:')
print(interplate_cv[interplate_cv['inter_plate_cv'] < 0.1])

In [ ]:
kdm.plot_cumulative_peptide_count_by_cv(interplate_cv)

In [ ]:
selected_peptides = kdm.get_lowest_cv_peptides(interplate_cv, 10)
selected_peptides

In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 
# using only peptides in the top 10% lowest inter-plate CV for normalization

pool_selected_df = skyline_pool[skyline_pool['Peptide Sequence'].isin(selected_peptides)]

# Remove bad plasmapool 

bad_pools = [
       'PRM_20260422_DA_4K_B2_Plate_7_C1.raw',
       'PRM_20260422_DA_4K_B2_Plate_8_B8.raw']


# Remove bad peptides 
bad_peptides = ['ESDTSYVSLK', 'VTLNGVPAQPLGPR', 'FSIEGSYQLEK', 'GLLSGWAR']

# Filter bad_pools from pool_selected_df
pool_selected_df = pool_selected_df[~pool_selected_df['File Name'].isin(bad_pools)]

# Filter bad_peptides from pool_selected_df
pool_selected_df = pool_selected_df[~pool_selected_df['Peptide Sequence'].isin(bad_peptides)]

# Filter pool_data to include only those peptides for normalization calculation
pool_selected_df

In [ ]:
kdm.plot_pool_boxplot(pool_selected_df)

In [ ]:
plate_factor_table, conversion_factors, model = kdm.get_plate_conversion_factors(pool_selected_df, col_plate='characteristics[plate]', log_transform=True  )

In [ ]:
plate_factor_table

In [ ]:
conversion_factors

In [ ]:
kdm.plot_plate_conversion_factors(pool_selected_df, col_plate='characteristics[plate]', log_transform=True  )

In [ ]:
# Use the adjust_ratio_by_plate function above to add RatioLightToHeavy_adj to skyline_merge
# First, construct a Plate column of the correct type for the function
skyline_merge_adj = skyline_merge.copy()
# Remove samples from Replicate column that are in qc_samples
skyline_merge_adj = skyline_merge_adj[~skyline_merge_adj['Replicate'].isin(qc_samples)]

# Remove sampels from Plate_1 from characteristics[plate]
skyline_merge_adj = skyline_merge_adj[skyline_merge_adj['characteristics[plate]'] != 'Plate_1']


In [ ]:
# Adjust ratio by conversion factor 
skyline_merge_adj = kdm.adjust_ratio_by_plate(skyline_merge_adj, conversion_factors)

skyline_merge_adj.head()

In [ ]:
# From skyline_merge_adj, filter to pool samples
pool_data_adj = skyline_merge_adj[skyline_merge_adj['characteristics[Sample]'] != 'PlasmaPool'].copy()

# Reorder by characteristics[Plate]
pool_data_adj = pool_data_adj.sort_values(by='characteristics[plate]')

pool_data_adj.head()


In [ ]:
kdm.plot_pool_boxplot(pool_data_adj)

In [ ]:
pool_selected_adj = pool_data_adj[pool_data_adj['Peptide Sequence'].isin(selected_peptides)]

kdm.plot_plate_conversion_factors(pool_selected_adj, col_plate='characteristics[plate]', log_transform=True  )

# SDRF metadata

Overview of metadata

In [ ]:
import matplotlib.pyplot as plt

# Make a barplot of 'factor value[disease]' column
plt.figure(figsize=(12,6))
value_counts = sdrf_data_file['factor value[disease]'].value_counts()
ax = value_counts.plot(kind='bar')
plt.xlabel('Disease')
plt.ylabel('Count')
plt.title("Sample count by 'factor value[disease]'")
plt.tight_layout()

# Add number on top of each bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', 
                (p.get_x() + p.get_width() / 2, p.get_height()), 
                ha='center', va='bottom', fontsize=10)

plt.show()

# Calculate absolute quantification

In [ ]:
# kdm.validate_sdrf(sdrf_path)

# Need to fix this 

In [ ]:
qreps_lot = sdrf_data.extract_qreps_lot_number()
print(qreps_lot)

In [ ]:
# Extract qRePS lot from SDRF file 
qreps_lot = sdrf_data.extract_qreps_lot_number()
# Fetch qRePS table
qreps_table = kdm.fetch_qreps_table(qreps_lot)

qreps_table.head()


In [ ]:
skyline_merge_adj.head()

In [ ]:
# Calculate absolute protein concentration
abs_df = kdm.get_absolute_conc(qreps_table, skyline_merge_adj)

In [ ]:
target_fasta = 'https://proteomedge.com/download/DE17501/DE17501_sequences.fasta'

# Read fasta file
fasta_file = kdm.fetch_fasta(qreps_lot)
fasta_file.head()

In [ ]:
kdm.map_peptide_sequence(abs_df, fasta_file, 'QR0330186_VWF|P04275')

In [ ]:
kdm.plot_median_peptide_concentration_by_group(
    abs_df,
    sdrf_data_file,
    protein_name="QR0290109_EGFR|P00533",
)

In [ ]:
# kdm.plot_all_median_peptide_concentration_by_group(
#     abs_df,
#     sdrf_data_file,
#     pdf_path="data/DA4000/median_peptide_concentration_by_group.pdf",
# )

In [ ]:
kdm.plot_peptide_concentration_by_group(
    abs_df,
    sdrf_data_file,
    color_col='characteristics[disease category]',
    protein_name="QR0290109_EGFR|P00533",
)

In [ ]:
import importlib, kardemumma as kdm
importlib.reload(kdm)

In [ ]:
kdm.plot_peptide_all(
    abs_df,
    sdrf_data_file,
    color_col='characteristics[disease category]',
    protein_name="QR0290109_EGFR|P00533",
)

In [ ]:
kdm.plot_all_all(
    abs_df,
    sdrf_data_file,
    color_col='characteristics[disease category]',
    pdf_path="data/DA4000/peptide_all_all_by_group.pdf",
)

In [ ]:
kdm.plot_all_peptide_concentration_by_group(
    abs_df=abs_df,
    sdrf_data_file=sdrf_data_file,
    color_col='characteristics[disease category]',
    pdf_path="data/DA4000/peptide_concentration_by_group.pdf",
)